# Computer Vision Image Pre-processing Tuning Guide (计算机视觉图像预处理调参指南)
This notebook contains the most common CV pre-processing methods. Each cell represents one method, allowing you to tune parameters and observe the effects in real-time.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

# Helper function to display images side-by-side
def show_images(img1, title1, img2, title2, cmap1=None, cmap2=None):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img1, cmap=cmap1)
    plt.title(title1)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img2, cmap=cmap2)
    plt.title(title2)
    plt.axis('off')
    plt.show()

# Download a standard sample image (Lena)
url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
req = urllib.request.urlopen(url)
arr = np.asarray(bytearray(req.read()), dtype=np.uint8)
image_bgr = cv2.imdecode(arr, -1)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB) # Convert to RGB for matplotlib

print("Setup Complete! Original Image Loaded:")
plt.imshow(image_rgb)
plt.axis('off')
plt.show()


In [ ]:
# ==========================================
# 1. Image Resizing / Scaling (图像缩放)
# ==========================================
# Purpose (用途): Standardizes image dimensions for model input, or reduces resolution for faster processing.
# Combined With (常配合使用的组合): Usually the very first step in Deep Learning (深度学习) pipelines.
# Specific Image Type (适用图像类型): All types. But beware of aspect ratio distortion for objects like text/documents.

# --- PARAMETERS TO TUNE (可调参数) ---
target_width = 128  # Target width in pixels. Larger = higher resolution. (目标宽度像素)
target_height = 128 # Target height in pixels. (目标高度像素)

# interpolation (插值方法): Method used to calculate pixel values for the new size.
#   - cv2.INTER_NEAREST: Fastest, low quality. Good for Segmentation Masks (分割掩码).
#   - cv2.INTER_LINEAR: Good balance of speed/quality. (Default)
#   - cv2.INTER_AREA: Best for shrinking/downsampling (缩小图像), avoids moire patterns.
#   - cv2.INTER_CUBIC: Slower, but highest quality for zooming in (放大图像).
interpolation_method = cv2.INTER_AREA 

resized_image = cv2.resize(image_rgb, (target_width, target_height), interpolation=interpolation_method)
show_images(image_rgb, f"Original {image_rgb.shape[:2]}", resized_image, f"Resized {resized_image.shape[:2]}")


In [ ]:
# ==========================================
# 2. Grayscale Conversion (灰度转换)
# ==========================================
# Purpose (用途): Reduces complexity from 3 color channels (RGB) to 1 intensity channel. 
# Combined With (常配合使用的组合): Canny Edge Detection (边缘检测), Thresholding (二值化), Histogram Equalization.
# Specific Image Type (适用图像类型): Document scans, X-Rays, Facial Recognition, or any task where color is irrelevant.

# --- PARAMETERS TO TUNE (可调参数) ---
# code: The color space conversion code.
#   - cv2.COLOR_RGB2GRAY: Converts RGB to Gray.
#   - cv2.COLOR_RGB2HSV: Converts RGB to HSV (Hue, Saturation, Value).
conversion_code = cv2.COLOR_RGB2GRAY

gray_image = cv2.cvtColor(image_rgb, conversion_code)
show_images(image_rgb, "Original RGB", gray_image, "Grayscale", cmap2='gray')


In [ ]:
# ==========================================
# 3. Gaussian Blur / Smoothing (高斯模糊 / 平滑)
# ==========================================
# Purpose (用途): Reduces high-frequency noise and detail. 
# Combined With (常配合使用的组合): Edge Detection (to prevent false edges from noise), Thresholding.
# Specific Image Type (适用图像类型): Noisy/grainy photos, low-light conditions.

# --- PARAMETERS TO TUNE (可调参数) ---
# ksize (卷积核大小): Tuple (width, height). MUST be positive and ODD (e.g., (3,3), (5,5), (11,11)).
#   - Effect: Larger kernel = stronger blur.
kernel_size = (9, 9) 

# sigmaX (X方向标准差): Standard deviation in X. 
#   - Effect: If 0, it is automatically calculated from kernel_size. Higher value = heavier blur.
sigmaX = 0 

blurred_image = cv2.GaussianBlur(image_rgb, kernel_size, sigmaX)
show_images(image_rgb, "Original", blurred_image, f"Gaussian Blur ksize={kernel_size}")


In [ ]:
# ==========================================
# 4. Bilateral Filtering (双边滤波)
# ==========================================
# Purpose (用途): Highly effective at noise removal WHILE preserving sharp edges (unlike Gaussian Blur).
# Combined With (常配合使用的组合): Cartoonifying pipelines, Facial Retouching (skin smoothing).
# Specific Image Type (适用图像类型): Portraits, images where object boundaries must remain crisp.

# --- PARAMETERS TO TUNE (可调参数) ---
# d: Diameter of pixel neighborhood. 
#   - Effect: > 5 is slow but effective. 9 is a good strong default.
d = 9 

# sigmaColor: Filter sigma in the color space.
#   - Effect: Larger value = colors farther apart will be mixed together. (Try 25 vs 150)
sigmaColor = 75 

# sigmaSpace: Filter sigma in the coordinate space.
#   - Effect: Larger value = farther pixels will influence each other (if colors are similar).
sigmaSpace = 75 

bilateral_image = cv2.bilateralFilter(image_rgb, d, sigmaColor, sigmaSpace)
show_images(image_rgb, "Original", bilateral_image, "Bilateral Filter (Edges Preserved)")


In [ ]:
# ==========================================
# 5. CLAHE (Contrast Limited Adaptive Histogram Equalization - 限制对比度自适应直方图均衡化)
# ==========================================
# Purpose (用途): Improves local contrast. Better than standard Equalization because it prevents over-amplifying noise.
# Combined With (常配合使用的组合): Applied to Grayscale images, before Thresholding.
# Specific Image Type (适用图像类型): Medical imaging (X-rays/MRIs), foggy/hazy scenes, unevenly lit documents.

# Note: CLAHE requires a 1-channel (grayscale) image.
gray_for_clahe = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# --- PARAMETERS TO TUNE (可调参数) ---
# clipLimit (对比度限制阈值): 
#   - Effect: Higher = more contrast, but potentially more noise. Try 1.0, 2.0, and 5.0.
clip_limit = 2.0 

# tileGridSize (网格大小): Divides image into tiles for local equalization.
#   - Effect: Smaller tiles = more localized contrast. (8, 8) is a standard default.
tile_grid_size = (8, 8)

clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
clahe_image = clahe.apply(gray_for_clahe)
show_images(gray_for_clahe, "Original Grayscale", clahe_image, "CLAHE Enhanced", cmap1='gray', cmap2='gray')


In [ ]:
# ==========================================
# 6. Otsu's Thresholding / Binarization (Otsu 二值化/阈值处理)
# ==========================================
# Purpose (用途): Converts grayscale image into a purely binary (Black & White) image. Otsu automatically finds the optimal threshold.
# Combined With (常配合使用的组合): Grayscale (Required), Gaussian Blur (Highly Recommended before this step), Morphology.
# Specific Image Type (适用图像类型): Document OCR (光学字符识别), foreground extraction, defect detection.

# Best Practice: Blur first to remove noisy speckles in the binary output
blurred_for_thresh = cv2.GaussianBlur(gray_for_clahe, (5, 5), 0)

# --- PARAMETERS TO TUNE (可调参数) ---
# maxval: The pixel value assigned to pixels passing the threshold.
max_value = 255 # Usually 255 for pure white.

# threshold_type: 
#   - cv2.THRESH_BINARY + cv2.THRESH_OTSU: Standard automatic threshold.
#   - cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU: Inverted (background black, text white).
threshold_type = cv2.THRESH_BINARY + cv2.THRESH_OTSU

# retval contains the optimal threshold value Otsu calculated
retval, binary_image = cv2.threshold(blurred_for_thresh, 0, max_value, threshold_type)

print(f"Optimal Threshold Value found by Otsu: {retval}")
show_images(blurred_for_thresh, "Blurred Grayscale", binary_image, "Otsu's Binary Threshold", cmap1='gray', cmap2='gray')


In [ ]:
# ==========================================
# 7. Canny Edge Detection (Canny 边缘检测)
# ==========================================
# Purpose (用途): Extracts structural outlines of objects.
# Combined With (常配合使用的组合): Grayscale (Required), Gaussian Blur (Crucial!), Hough Transforms.
# Specific Image Type (适用图像类型): Lane detection, contour extraction, structural analysis.

# Best Practice: Always use blurred grayscale image
gray_blurred_for_canny = cv2.GaussianBlur(gray_for_clahe, (5, 5), 0)

# --- PARAMETERS TO TUNE (可调参数) ---
# threshold1 (Lower Bound): Edges with gradient below this are discarded.
#   - Effect: Lowering this finds more faint lines and noise. Try 20 vs 100.
lower_threshold = 50 

# threshold2 (Upper Bound): Edges with gradient above this are strictly kept as edges.
#   - Effect: Increasing this makes edge detection stricter. Try 150 vs 250.
#   - Rule of Thumb: Ratio of T1:T2 should be 1:2 or 1:3.
upper_threshold = 150 

edges = cv2.Canny(gray_blurred_for_canny, lower_threshold, upper_threshold)
show_images(image_rgb, "Original", edges, f"Canny Edges (L:{lower_threshold}, U:{upper_threshold})", cmap2='gray')


In [ ]:
# ==========================================
# 8. Morphological Operations: Dilation & Erosion (形态学操作：膨胀与腐蚀)
# ==========================================
# Purpose (用途): Processes shapes in binary images. Used to clean up noise or connect broken lines.
#   - Dilation (膨胀): Thickens lines, fills small holes.
#   - Erosion (腐蚀): Shrinks lines, removes small white noise dots.
# Combined With (常配合使用的组合): Applied AFTER Thresholding or Canny Edge Detection.
# Specific Image Type (适用图像类型): Binary masks, OCR text cleanup.

# --- PARAMETERS TO TUNE (可调参数) ---
# kernel_size (结构元素大小): Defines the neighborhood to consider.
#   - Effect: Larger kernel = much stronger thickening/shrinking. Try (3,3) vs (9,9).
kernel_size = (5, 5)
kernel = np.ones(kernel_size, np.uint8)

# iterations (迭代次数): How many times the operation is applied sequentially.
#   - Effect: More iterations = compounded effect.
iterations = 1 

# Apply Dilation to the Canny edges to make them thicker
dilated_edges = cv2.dilate(edges, kernel, iterations=iterations)

show_images(edges, "Original Canny Edges", dilated_edges, f"Dilated Edges (kernel {kernel_size})", cmap1='gray', cmap2='gray')


In [ ]:
# ==========================================
# 9. Normalization / Min-Max Scaling (归一化)
# ==========================================
# Purpose (用途): Scales pixel values (usually 0-255) to a specific range like [0.0, 1.0].
# Combined With (常配合使用的组合): The final step before feeding an image into a Neural Network (神经网络).
# Specific Image Type (适用图像类型): Any image used for Deep Learning (ResNet, YOLO, etc.) to prevent exploding gradients.

# --- PARAMETERS TO TUNE (可调参数) ---
# This is a mathematical transformation, no complex OpenCV parameters.

# Option 1: Scale to [0, 1]
normalized_01 = image_rgb.astype('float32') / 255.0

# Option 2: Z-Score Standardization (Standard Normal Distribution - mean 0, std 1)
mean = np.mean(image_rgb, axis=(0, 1))
std = np.std(image_rgb, axis=(0, 1))
# Adding a tiny epsilon (1e-7) prevents division by zero
normalized_zscore = (image_rgb.astype('float32') - mean) / (std + 1e-7)

print(f"Original Image Range: Min {image_rgb.min()}, Max {image_rgb.max()}")
print(f"Normalized [0,1] Range: Min {normalized_01.min():.2f}, Max {normalized_01.max():.2f}")
print(f"Z-Score Range: Min {normalized_zscore.min():.2f}, Max {normalized_zscore.max():.2f}")

show_images(image_rgb, "Original", normalized_01, "Normalized [0, 1]")
